In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/final-data/cleaned_products_with_captions.csv


In [2]:
!pip install pymilvus

In [3]:
!pip install rank_bm25 --quiet

In [4]:
import os
import pickle
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from typing import Dict, List, Tuple, Any, Optional
import traceback

# Ensure NLTK resources are available
def ensure_nltk_resources():
    """Ensure all required NLTK resources are downloaded."""
    required_resources = [
        ('punkt', 'tokenizers/punkt'),
        ('stopwords', 'corpora/stopwords')
    ]
    
    for resource, path in required_resources:
        try:
            nltk.data.find(path)
        except LookupError:
            print(f"Downloading NLTK resource '{resource}'...")
            nltk.download(resource, quiet=True)

# Cache for BM25 models
_BM25_CACHE = {}  # Key: collection_name, Value: (bm25_model, corpus_mapping)

def get_bm25_model(collection_name: str, data_path: str) -> Tuple[Any, Dict]:
    """
    Load BM25 model for a collection if not already loaded.
    
    Args:
        collection_name: Name of the collection
        data_path: Path to data directory
        
    Returns:
        Tuple of (bm25_model, corpus_mapping)
    """
    global _BM25_CACHE
    
    # Return cached model if available
    if collection_name in _BM25_CACHE:
        print(f"Using cached BM25 model for collection '{collection_name}'")
        return _BM25_CACHE[collection_name]
    
    # Paths to BM25 files
    bm25_path = os.path.join(data_path, f"{collection_name}_bm25.pkl")
    corpus_map_path = os.path.join(data_path, f"{collection_name}_corpus_map.pkl")
    
    # Load BM25 model if files exist
    if os.path.exists(bm25_path) and os.path.exists(corpus_map_path):
        try:
            with open(bm25_path, 'rb') as f:
                bm25_model = pickle.load(f)
            with open(corpus_map_path, 'rb') as f:
                corpus_mapping = pickle.load(f)
            
            # Cache the loaded models
            _BM25_CACHE[collection_name] = (bm25_model, corpus_mapping)
            print(f"BM25 model for collection '{collection_name}' loaded and cached")
            return _BM25_CACHE[collection_name]
        except Exception as e:
            print(f"Error loading BM25 model for collection '{collection_name}': {e}")
    
    return None, None

def clear_bm25_cache(collection_name: str = None):
    """
    Clear BM25 model from cache.
    
    Args:
        collection_name: Specific collection to clear, or None to clear all
    """
    global _BM25_CACHE
    
    if collection_name:
        if collection_name in _BM25_CACHE:
            del _BM25_CACHE[collection_name]
            print(f"Cleared cached BM25 model for collection '{collection_name}'")
    else:
        _BM25_CACHE.clear()
        print("Cleared all cached BM25 models")

def create_bm25_index(collection_name: str, data_path: str, texts: List[str], 
                      product_ids: List[str]) -> bool:
    """
    Create and save BM25 index for a collection.
    
    Args:
        collection_name: Name of the collection
        data_path: Path to data directory
        texts: List of processed text documents
        product_ids: List of product IDs corresponding to texts
        
    Returns:
        Success status (bool)
    """
    ensure_nltk_resources()
    stop_words = set(stopwords.words('english'))
    
    try:
        print("Starting BM25 tokenization...")
        tokenized_corpus = []
        for text in texts:
            tokens = word_tokenize(text.lower())
            tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
            tokenized_corpus.append(tokens)
        
        print(f"Creating BM25 model with {len(tokenized_corpus)} documents")
        bm25 = BM25Okapi(tokenized_corpus)
        print("BM25 model created successfully")
        
        # Save BM25 model and corpus mapping to disk for later use
        bm25_path = os.path.join(data_path, f"{collection_name}_bm25.pkl")
        corpus_map_path = os.path.join(data_path, f"{collection_name}_corpus_map.pkl")
        
        # Save mapping between corpus index and product ID
        corpus_map = {i: product_ids[i] for i in range(len(product_ids))}
        
        with open(bm25_path, 'wb') as f:
            pickle.dump(bm25, f)
        
        with open(corpus_map_path, 'wb') as f:
            pickle.dump(corpus_map, f)
        
        # Update cache
        _BM25_CACHE[collection_name] = (bm25, corpus_map)
        
        print(f"BM25 index saved to {bm25_path}")
        return True
    except Exception as e:
        print(f"Error during BM25 creation: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

def search_bm25(collection_name: str, query: str, 
                bm25_model: Any, corpus_mapping: Dict, top_k: int = 10) -> Dict[str, float]:
    """
    Search using BM25 model.
    
    Args:
        collection_name: Name of the collection
        query: User query string
        bm25_model: BM25 model
        corpus_mapping: Mapping from corpus index to product ID
        top_k: Number of top results to return
        
    Returns:
        Dictionary mapping product_id to BM25 score
    """
    ensure_nltk_resources()
    results_dict = {}
    
    try:
        # Tokenize and preprocess query for BM25
        stop_words = set(stopwords.words('english'))
        
        # Preprocess query
        query_tokens = word_tokenize(query.lower())
        query_tokens = [t for t in query_tokens if t not in stop_words and len(t) > 2]
        
        # Get BM25 scores
        if query_tokens:
            bm25_scores = bm25_model.get_scores(query_tokens)
            
            # Get top BM25 results
            bm25_top_indices = sorted(range(len(bm25_scores)), 
                                     key=lambda i: bm25_scores[i], 
                                     reverse=True)[:top_k]
            
            # Map indices to product IDs
            for idx in bm25_top_indices:
                if idx in corpus_mapping:
                    product_id = corpus_mapping[idx]
                    results_dict[product_id] = bm25_scores[idx]
            
            print(f"BM25 found {len(results_dict)} matching products")
    except Exception as e:
        print(f"Error in BM25 search: {e}")
    
    return results_dict

In [5]:
!pip install sentence-transformers pandas tqdm

In [6]:
!pip install -U sentence-transformers

In [7]:
from pathlib import Path
import json
DATA_PATH = Path("/kaggle/working/bm25_index")  # or a valid path
DATA_PATH.mkdir(parents=True, exist_ok=True)

In [8]:
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType, utility

MILVUS_HOST = "34.227.82.106"   
MILVUS_PORT = "19530"

In [9]:
"""
Enhanced ShopTalk Implementation
Optimized based on actual dataset analysis and common query patterns
"""

#  IMPORTS AND CONFIGURATION
# Standard library imports
import os
import gc
import re
import json
from openai import OpenAI
# from openai import ChatCompletion
from typing import List, Dict, Any



from pydantic import BaseModel, Field


# Data processing libraries
import pandas as pd
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# NLP and ML libraries
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from transformers import pipeline

# Vector database
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType, utility
from pymilvus import connections
import sys
from multiprocessing import Pool, cpu_count
import warnings
warnings.filterwarnings("ignore")

2025-04-11 06:08:10.485489: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744351690.509081     611 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744351690.516629     611 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [10]:
input_dir = '/kaggle/input/final-data'
file_path = f'{input_dir}/cleaned_products_with_captions.csv'   
df = pd.read_csv(file_path)
#OPENAI_API_KEY="test"    

In [ ]:
#sampled_df = df.sample(n=10000, random_state=42)

In [ ]:
#sampled_df.head(5)

In [11]:

df.head(5)

,item_id,image_id,image_path,item_name,product_type,bullet_point,item_keywords,product_description,image_caption,brand,model_name,color_standardized,style,material,fabric_type,pattern,item_shape,finish_type,node_name
0,B07T7ZY6P6,71bcNwxkB7L,/c3/c3406c06.jpg,Amazon Brand - Solimo Designer Lighting 3D Pri...,CELLULAR_PHONE_CASE,3D Printed Hard Back Case Mobile Cover for Viv...,mobile cover | back cover | mobile case | phon...,NaN,a phone case with christmas lights on it,Amazon Brand - Solimo,Vivo V1 Max,multi-colored,NaN,NaN,NaN,NaN,NaN,NaN,/Categories/Mobiles & Accessories/Mobile Acces...
1,B07YQPTXFK,71I4ezaoNgL,/10/102157fb.jpg,"Amazon Meal Kits, Chicken & Vegetable Stir-Fry...",GROCERY,Ginger chicken stir-fried with vegetables serv...,dinner | dinners | dinner kit | easy dinner | ...,NaN,a box of chicken and vegetable pasta on a whit...,Amazon Meal Kits,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Categories/Fresh Meal Kits
2,B0853X5SQ5,616ZijT0A5L,/a7/a7270e7a.jpg,Amazon Brand - Solimo Designer Red Moon UV Pri...,CELLULAR_PHONE_CASE,"Snug fit for Lava Z61, with perfect cut-outs f...",Back Cover | Designer Case | Designer Red Moon...,NaN,a phone case with the silhouette of a red moon...,Amazon Brand - Solimo,Lava Z61,multi-colored,NaN,Silicon,NaN,NaN,NaN,NaN,/Categories/Mobiles & Accessories/Mobile Acces...
3,B07PPXFC2R,51rXStEHElL,/55/55c7d8e2.jpg,Amazon Essentials Belice Women's Flat Ballet F...,SHOES,Classic and versatile ballet flats for everyda...,NaN,NaN,a pair of leopard print shoes,NaN,Belice Women's Ballet Flat,NaN,NaN,NaN,100% Synthetic,NaN,NaN,NaN,/Kategorien/Schuhe/Damen/Flache Schuhe/Ballerinas
4,B07VTCS9DQ,71byIpH+1LL,/87/87868d71.jpg,Amazon Brand - Solimo Deep Moisturizing Baby W...,SKIN_CLEANING_AGENT,One 13-fluid ounce bottle of deep moisturizing...,baby wash | baby body wash | bath wash for bab...,NaN,a bottle of soap on a white background,Solimo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# config_fields.py
"""
Configuration file for field ordering and properties in the ShopTalk application.
This centralizes field definitions to make experimentation easier.
"""
import os
#from config_embedding import EMBEDDING_DIMENSION

# Define field structure with ordering and max_length properties
FIELDS_CONFIG = {
    # ID fields (will be processewwd first)
    'product_id': {'order': 1, 'max_length': 100, 'is_id': True, 'is_primary': True},
    'primary_image_id': {'order': 2, 'max_length': 100, 'is_id': True},
    'image_storage_path': {'order': 3, 'max_length': 100, 'is_id': True},
    
    # Core embedding fields in priority order
    'product_title': {'order': 4, 'max_length': 500, 'is_id': False},
    'product_category': {'order': 5, 'max_length': 100, 'is_id': False},
    'product_features': {'order': 6, 'max_length': 4000, 'is_id': False},
    'long_description': {'order': 7, 'max_length': 4000, 'is_id': False},
    'visual_description': {'order': 8, 'max_length': 500, 'is_id': False},
    'manufacturer_brand': {'order': 9, 'max_length': 100, 'is_id': False},
    'product_model': {'order': 10, 'max_length': 200, 'is_id': False},
    'category_path': {'order': 11, 'max_length': 500, 'is_id': False},
    'primary_color': {'order': 12, 'max_length': 100, 'is_id': False},
    'design_style': {'order': 13, 'max_length': 100, 'is_id': False},
    'primary_material': {'order': 14, 'max_length': 100, 'is_id': False},
    'textile_material': {'order': 15, 'max_length': 100, 'is_id': False},
    'design_pattern': {'order': 16, 'max_length': 100, 'is_id': False},
    'physical_shape': {'order': 17, 'max_length': 100, 'is_id': False},
    'surface_finish': {'order': 18, 'max_length': 100, 'is_id': False},
    'search_terms': {'order': 19, 'max_length': 4000, 'is_id': False},
}

# Generate lists based on the config for easier access
ID_COLUMNS = [field for field, props in sorted(
    [(f, p) for f, p in FIELDS_CONFIG.items() if p['is_id']], 
    key=lambda x: x[1]['order']
)]

EMBEDDING_PRIORITY_COLUMNS = [field for field, props in sorted(
    [(f, p) for f, p in FIELDS_CONFIG.items() if not p['is_id']], 
    key=lambda x: x[1]['order']
)]

# Get all fields in schema order
ALL_FIELDS = [field for field, props in sorted(
    FIELDS_CONFIG.items(), 
    key=lambda x: x[1]['order']
)]

# Field mapping for backward compatibility with original ABO dataset names
ORIGINAL_TO_NEW_MAPPING = {
    "item_id": "product_id",
    "image_id": "primary_image_id",
    "image_path": "image_storage_path",
    # Order of the fields
    "item_name": "product_title",
    "product_type": "product_category",
    "bullet_point": "product_features",
    "product_description": "long_description",
    "image_caption": "visual_description",
    "brand": "manufacturer_brand",
    "model_name": "product_model",
    "node_name": "category_path",
    "color_standardized": "primary_color",
    "style": "design_style",
    "material": "primary_material",
    "fabric_type": "textile_material",
    "pattern": "design_pattern",
    "item_shape": "physical_shape",
    "finish_type": "surface_finish",
    "item_keywords": "search_terms",
}

# Reverse mapping for backward compatibility
NEW_TO_ORIGINAL_MAPPING = {v: k for k, v in ORIGINAL_TO_NEW_MAPPING.items()}

# Function to modify field order during experimentation
def reorder_fields(new_order_list):
    """
    Temporarily reorder fields for experimentation.
    
    Args:
        new_order_list: List of field names in desired order
    
    Returns:
        Original order to restore later
    """
    global EMBEDDING_PRIORITY_COLUMNS
    old_order = EMBEDDING_PRIORITY_COLUMNS.copy()
    
    # Update only non-ID fields
    non_id_fields = [field for field in new_order_list 
                    if field in FIELDS_CONFIG and not FIELDS_CONFIG[field]['is_id']]
    
    EMBEDDING_PRIORITY_COLUMNS = non_id_fields
    return old_order

In [13]:
# config_embedding.py
import os

# -------------------------------
# ✏️ MANUALLY SELECT ONE MODEL HERE:
# Uncomment exactly one line to activate a model.
# Each line includes the reason why you'd use it.

DEFAULT_MODEL = "bge_small"       # ✅ Best trade-off between speed and quality. Great for general-purpose vector search and fast inference.
# DEFAULT_MODEL = "miniLM"          # ⚡ Fastest model. Ideal for prototypes, low-resource environments, or real-time retrieval where latency matters.
# DEFAULT_MODEL = "multiqa_mpnet"   # 🧠 Fine-tuned for QA-style search. Excellent for retrieval tasks where user queries are full sentences/questions.
# DEFAULT_MODEL = "all_mpnet"       # 🔍 High semantic quality. Great all-rounder for diverse query types, but slower than MiniLM.
# DEFAULT_MODEL = "bge_large"       # 🥇 Superior quality. Use when quality is top priority and you have compute/GPU to support it.
# DEFAULT_MODEL = "e5_large"          # 🏆 Highest quality. Best for long documents and highly nuanced semantic retrieval. Slowest but most powerful.
# -------------------------------


# Models ordered by ranking (real-time performance balanced with quality)
MODEL_CHOICES = {
    # Best balance of speed and quality
    "bge_small": {
        "name": "BAAI/bge-small-en-v1.5",
        "dimension": 384,
        "metric": "IP",
        "rank": 1
    },
    # Fastest but lower quality
    "miniLM": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "dimension": 384,
        "metric": "IP",
        "rank": 2
    },
    # Good quality-speed middle ground
    "multiqa_mpnet": {
        "name": "sentence-transformers/multi-qa-mpnet-base-dot-v1",
        "dimension": 768,
        "metric": "IP",
        "rank": 3
    },
    # Higher quality but slower
    "all_mpnet": {
        "name": "sentence-transformers/all-mpnet-base-v2",
        "dimension": 768,
        "metric": "IP", 
        "rank": 4
    },
    # Best quality but significantly slower queries
    "bge_large": {
        "name": "BAAI/bge-large-en-v1.5",
        "dimension": 1024,
        "metric": "IP",
        "rank": 5
    },
    # Highest quality but slowest
    "e5_large": {
        "name": "intfloat/e5-large-v2",
        "dimension": 1024,
        "metric": "IP",
        "rank": 6
    }
}

# Use environment variable if set; fallback to default
SELECTED_MODEL_ALIAS = os.getenv("EMBED_MODEL_ALIAS", DEFAULT_MODEL)

# If an invalid model is specified, use the default
if SELECTED_MODEL_ALIAS not in MODEL_CHOICES:
    print(f"Warning: Unknown model alias '{SELECTED_MODEL_ALIAS}'. Using default: {DEFAULT_MODEL}")
    SELECTED_MODEL_ALIAS = DEFAULT_MODEL

# Extract all needed properties from the selected model configuration
EMBEDDING_MODEL_NAME = MODEL_CHOICES[SELECTED_MODEL_ALIAS]["name"]
EMBEDDING_DIMENSION = MODEL_CHOICES[SELECTED_MODEL_ALIAS]["dimension"]
SIMILARITY_METRIC = MODEL_CHOICES[SELECTED_MODEL_ALIAS]["metric"]

# For debugging/logging
MODEL_RANK = MODEL_CHOICES[SELECTED_MODEL_ALIAS]["rank"]# 

# Optional debug print
print(f"Model Loaded: {EMBEDDING_MODEL_NAME} | dim: {EMBEDDING_DIMENSION} | metric: {SIMILARITY_METRIC}")
FIELD_NAME_MAPPING = ORIGINAL_TO_NEW_MAPPING
REVERSE_FIELD_MAPPING = NEW_TO_ORIGINAL_MAPPING

Model Loaded: BAAI/bge-small-en-v1.5 | dim: 384 | metric: IP


In [14]:
!pip install hf_xet

In [15]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
#llm = pipeline("text2text-generation", model="google/flan-t5-base",device=device)
# llm = pipeline("text2text-generation", model="google/flan-t5-large")

# Initialize embedding model - using BGE for better semantic search performance
# EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
# EMBEDDING_DIMENSION = 384
# SIMILARITY_METRIC = "cosine"
print(f"Using embedding model: {EMBEDDING_MODEL_NAME} with dim {EMBEDDING_DIMENSION}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
if torch.cuda.is_available():
    embedding_model = embedding_model.to("cuda")
# Field name mapping for backward compatibility


FIELD_NAME_MAPPING = ORIGINAL_TO_NEW_MAPPING
REVERSE_FIELD_MAPPING = NEW_TO_ORIGINAL_MAPPING

# All fields for collection schema
ALL_FIELDS = list(FIELDS_CONFIG.keys())

Using embedding model: BAAI/bge-small-en-v1.5 with dim 384


In [16]:
def ensure_nltk_resources():
    """
    Ensure all required NLTK resources are downloaded.
    This centralized function should be called before any NLTK operations.
    """
    import nltk
    
    # Define resources needed throughout the application
    required_resources = [
        ('punkt', 'tokenizers/punkt'),
        ('stopwords', 'corpora/stopwords')
    ]
    
    # Check and download each resource if not already present
    for resource, path in required_resources:
        try:
            nltk.data.find(path)
            print(f"NLTK resource '{resource}' is already available.")
        except LookupError:
            print(f"Downloading NLTK resource '{resource}'...")
            nltk.download(resource, quiet=True)
            print(f"NLTK resource '{resource}' has been downloaded.")

# Initialize NLTK resources
ensure_nltk_resources()
print("NLTK resources initialized")

# # Call this after model initialization
# test_result = test_llm()
# print("LLM test completed")

#  DATA PREPROCESSING AND VALIDATION
# Field Validation
def validate_field_length(value, max_length):
    """Truncate string values that exceed the maximum field length"""
    if value is None:
        return ""
    str_value = str(value)
    if len(str_value) > max_length:
        print(f"Truncating field value from {len(str_value)} to {max_length} characters")
        return str_value[:max_length]
    return str_value

# Convert old field names to new field names
def convert_field_names(df):
    """Convert dataframe columns from old field names to new field names"""
    renamed_columns = {}
    for old_name, new_name in FIELD_NAME_MAPPING.items():
        if old_name in df.columns:
            renamed_columns[old_name] = new_name
    
    # Rename only columns that exist in the dataframe
    if renamed_columns:
        df = df.rename(columns=renamed_columns)
    
    return df

# EMBEDDING GENERATION
# Text embedding
def generate_embeddings(text_list, batch_size=32):
    """Generate embeddings using Sentence Transformer."""
    # Handle empty text
    cleaned_texts = [text if text and str(text).lower() != "nan" else "empty" for text in text_list]
    
    # Generate embeddings with sentence-transformers
    embeddings = embedding_model.encode(
        cleaned_texts, 
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    return embeddings

# Enhanced document structure - Optimized for actual product distribution
def create_structured_document(product):
    """
    Create structured document representation optimized for actual product distribution.
    Uses field ordering from config_fields.py but new field names.
    """
    parts = []
    
    # Iterate through fields in priority order using config
    for field in EMBEDDING_PRIORITY_COLUMNS:
        # Try to get value using both new and original field names
        field_value = None
        if field in product:
            field_value = product[field]
        elif field in REVERSE_FIELD_MAPPING and REVERSE_FIELD_MAPPING[field] in product:
            field_value = product[REVERSE_FIELD_MAPPING[field]]
            
        # Skip empty values
        if not field_value or str(field_value).lower() == 'nan':
            continue
            
        # Format based on field type
        if field == 'product_title':
            parts.append(f"Title: {field_value} {field_value}")
        elif field == 'product_category':
            parts.append(f"Category: {field_value} {field_value} {field_value}")
        elif field == 'search_terms':
            # Truncate keywords to avoid dominating the embedding
            truncated_text = str(field_value)
            if len(truncated_text) > 1000:
                truncated_text = truncated_text[:1000]
            parts.append(f"Keywords: {truncated_text}")
        elif field == 'product_features':
            # Truncate features if too long
            features_text = str(field_value)
            if len(features_text) > 1500:
                features_text = features_text[:1500]
            parts.append(f"Features: {features_text}")
        elif field == 'visual_description':
            if field_value != "[undetermined product]":
                parts.append(f"Visual: {field_value}")
        elif field == 'manufacturer_brand':
            parts.append(f"Brand: {field_value}")
        elif field == 'primary_color':
            parts.append(f"Color: {field_value}")
        elif field == 'primary_material':
            parts.append(f"Material: {field_value}")
        elif field == 'design_style':
            parts.append(f"Style: {field_value}")
        else:
            # Generic field handling
            field_name = field.replace('_', ' ').title()
            parts.append(f"{field_name}: {field_value}")
    
    return " ".join(parts)



# # DOCUMENT PROCESSING
# # Main Processing Function
# def process_documents(collection: Collection, df: pd.DataFrame, DATA_PATH:str ):
#     """
#     Process product data from CSV files and insert into Milvus.
#     Creates structured embeddings with field prefixes following the priority order.
#     Also creates BM25 index for hybrid search.
#     """
#     print("process_documents() has been triggered")
#     print(f"Loaded {len(df)} products from CSV.")
    
#     # Convert old field names to new field names
#     df = convert_field_names(df)
    
       
#     # Load all fields into a dictionary for easier handling
#     field_data = {}
    
#     # Load all fields from the dataframe
#     for field in ALL_FIELDS:
#         field_data[field] = df[field].astype(str).tolist() if field in df.columns else [""] * len(df)
    
#     # Create structured text for embedding using the enhanced document structure
#     print("Creating structured text for embeddings...")
#     structured_texts = []
    
#     # Prepare data for BM25
#     bm25_texts = []
#     product_ids = field_data['product_id']
    
#     for i in range(len(field_data['product_id'])):
#         # Create a temporary product dictionary for this item
#         product = {field: field_data[field][i] for field in ALL_FIELDS}
        
#         # Use the enhanced document structure for vector search
#         structured_text = create_structured_document(product)
#         structured_texts.append(structured_text)
        
#         # Create a plain text representation for BM25
#         bm25_parts = []
#         # Add fields with appropriate weighting
#         if product.get('product_category'):
#             bm25_parts.append(f"{product['product_category']} {product['product_category']} {product['product_category']}")
#         if product.get('product_title'):
#             bm25_parts.append(f"{product['product_title']} {product['product_title']}")
#         if product.get('manufacturer_brand'):
#             bm25_parts.append(f"{product['manufacturer_brand']} {product['manufacturer_brand']}")
#         if product.get('primary_color'):
#             bm25_parts.append(f"{product['primary_color']} {product['primary_color']}")
#         if product.get('product_features') and str(product['product_features']).lower() != 'nan':
#             bm25_parts.append(str(product['product_features'])[:300])  # Truncate features
            
#         bm25_texts.append(" ".join(bm25_parts))
        
#         # Print sample of the first item to verify
#         if i == 0:
#             print(f"Sample structured text: {structured_text[:200]}...")
#             print(f"Sample BM25 text: {bm25_texts[0][:200]}...")
    
#     # Create BM25 index with the handler
#     if not bm25_index_exists(DATA_PATH):
#         print("Creating BM25 index...")
    
#         success = create_bm25_index(
#             collection_name=collection_name,
#             data_path=DATA_PATH,
#             texts=bm25_texts,
#             product_ids=product_ids
#         )
#         if not success:
#             print("Continuing without BM25 index...")
#     else:
#         print("BM25 index already exists. Skipping creation.")
#     # Generate embeddings for all products
#     print("Generating embeddings...")
#     embeddings = generate_embeddings(structured_texts, batch_size=32)
    
#     # Clear memory
#     print("Clearing memory...")
#     del structured_texts
#     gc.collect()
    
#     # Add validation step to prevent max length errors
#     print("Validating field lengths before insertion...")
#     for field in ALL_FIELDS:
#         if field in FIELDS_CONFIG:
#             max_length = FIELDS_CONFIG[field].get('max_length', 100)
#             field_data[field] = [validate_field_length(value, max_length) for value in field_data[field]]
        
#     # Prepare data for insertion according to schema order
#     data = []
    
#     # Add all fields in schema order
#     for field in ALL_FIELDS:
#         data.append(field_data[field])
    
#     # Add embeddings last
#     data.append(embeddings)
    
#     # Insert data into Milvus
#     print(f"Inserting {len(field_data['product_id'])} products into Milvus...")
#     collection.insert(data)
#     print("Data inserted successfully.")
    
#     # Create index for vector search
#     print("Creating index on embedding field...")
#     index_params = {
#         "metric_type": SIMILARITY_METRIC,  
#         "index_type": "IVF_FLAT",
#         "params": {"nlist": 128}
#     }
#     collection.create_index(field_name="embedding", index_params=index_params)
#     print("Index created successfully.")
    
#     # Load collection for searching
#     collection.load()
#     print(f"Collection '{collection_name}' loaded and ready for queries.")



NLTK resource 'punkt' is already available.
NLTK resource 'stopwords' is already available.
NLTK resources initialized


In [17]:
def prepare_bm25_data(df: pd.DataFrame):
    bm25_texts = []
    product_ids = df["product_id"].astype(str).tolist() if "product_id" in df.columns else [""] * len(df)

    for i in range(len(df)):
        product = df.iloc[i].to_dict()
        bm25_parts = []

        # Weighted fields
        if product.get("product_category"):
            bm25_parts.append(f"{product['product_category']} " * 3)
        if product.get("product_title"):
            bm25_parts.append(f"{product['product_title']} " * 2)
        if product.get("manufacturer_brand"):
            bm25_parts.append(f"{product['manufacturer_brand']} " * 2)
        if product.get("primary_color"):
            bm25_parts.append(f"{product['primary_color']} " * 2)
        if product.get("product_features") and str(product['product_features']).lower() != "nan":
            bm25_parts.append(str(product["product_features"])[:300])

        bm25_text = " ".join(bm25_parts).strip()
        bm25_texts.append(bm25_text)

        if i == 0:
            print(f"Sample BM25 text: {bm25_text[:200]}...")

    return bm25_texts, product_ids


In [18]:
def process_documents(
    df: pd.DataFrame,
    DATA_PATH: str,
    bm25_texts_all: list,
    product_ids_all: list
):
    print("process_documents() has been triggered")
    print(f"Loaded {len(df)} products from chunk.")

    # Normalize field names
    df = convert_field_names(df)

    # Prepare field data
    field_data = {
        field: df[field].astype(str).tolist() if field in df.columns else [""] * len(df)
        for field in ALL_FIELDS
    }

    # Create structured texts
    print("Creating structured text for embeddings...")
    structured_texts = []
    for i in range(len(df)):
        product = {field: field_data[field][i] for field in ALL_FIELDS}
        structured_text = create_structured_document(product)
        structured_texts.append(structured_text)
        if i == 0:
            print(f"Sample structured text: {structured_text[:200]}...")

    # Generate embeddings
    print("Generating embeddings...")
    embeddings = generate_embeddings(structured_texts, batch_size=32)

    # Clean memory
    del structured_texts
    gc.collect()

    # Validate field lengths
    print("Validating field lengths before insertion...")
    for field in ALL_FIELDS:
        if field in FIELDS_CONFIG:
            max_length = FIELDS_CONFIG[field].get('max_length', 100)
            field_data[field] = [
                validate_field_length(value, max_length) for value in field_data[field]
            ]

    # Prepare and insert into Milvus
    data = [field_data[field] for field in ALL_FIELDS] + [embeddings]

    print(f"Inserting {len(field_data['product_id'])} products into Milvus...")
    collection.insert(data)
    print("Data inserted successfully.")

    print("Creating index on embedding field...")
    index_params = {
        "metric_type": SIMILARITY_METRIC,
        "index_type": "IVF_FLAT",
        "params": {"nlist": 128}
    }
    collection.create_index(field_name="embedding", index_params=index_params)
    collection.load()
    #print(f"Collection '{collection_name}' loaded and ready for queries.")

    # Append BM25 batch data
    bm25_texts_batch, product_ids_batch = prepare_bm25_data(df)
    bm25_texts_all.extend(bm25_texts_batch)
    product_ids_all.extend(product_ids_batch)


In [19]:
from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility

# Collection Creation
def create_milvus_collection(collection_name: str):
    """Creates or gets a Milvus collection with proper schema for product data."""
    fields = []
    
    # Add all fields according to the config
    for field_name in ALL_FIELDS:
        props = FIELDS_CONFIG[field_name]
        
        if props.get('is_primary', False):
            fields.append(FieldSchema(
                name=field_name,
                dtype=DataType.VARCHAR,
                max_length=props['max_length'],
                is_primary=True
            ))
        else:
            fields.append(FieldSchema(
                name=field_name,
                dtype=DataType.VARCHAR,
                max_length=props['max_length']
            ))
    
    # Add embedding field last
    fields.append(FieldSchema(
        name="embedding",
        dtype=DataType.FLOAT_VECTOR,
        dim=EMBEDDING_DIMENSION
    ))
    
    schema = CollectionSchema(fields, "Product collection with embeddings ordered by priority")

    # Drop collection if it exists with a different schema
    if utility.has_collection(collection_name):
        print(f"Collection '{collection_name}' already exists. Dropping...")
        utility.drop_collection(collection_name)

    # Create new collection
    collection = Collection(name=collection_name, schema=schema)
    print(f"Collection '{collection_name}' created.")
    
    return collection


In [20]:
# Initialize NLTK resources
ensure_nltk_resources()
print("NLTK resources initialized")

NLTK resource 'punkt' is already available.
NLTK resource 'stopwords' is already available.
NLTK resources initialized


In [21]:
def connect_to_milvus():
    connections.connect(host=MILVUS_HOST, port=MILVUS_PORT)
    print("Connected to Milvus")

In [22]:
connect_to_milvus()

Connected to Milvus


In [23]:
collection_name="productsFinal"        
# Create or get the collection
collection = create_milvus_collection(collection_name)

Collection 'productsFinal' already exists. Dropping...
Collection 'productsFinal' created.


In [ ]:
# def create_or_reset_collection_once(collection_name: str):
#     return create_milvus_collection(collection_name)

In [24]:
def bm25_index_exists(data_path: str, index_filename: str = "bm25_index.pkl") -> bool:
    """
    Checks whether the BM25 index file already exists at the given path.

    Args:
        data_path (str): The directory path where the index is stored.
        index_filename (str): The expected file name of the BM25 index.

    Returns:
        bool: True if the index file exists, False otherwise.
    """
    index_path = os.path.join(data_path, index_filename)
    return os.path.isfile(index_path)


In [25]:
def process_documents_in_chunks(
    df: pd.DataFrame,
    DATA_PATH: str,
    num_parts: int = 10
):
    chunk_size = len(df) // num_parts + 1

    # Shared BM25 storage
    all_bm25_texts = []
    all_product_ids = []

    # You can create/reset collection here if needed
    # global collection_name
    # collection = create_or_reset_collection_once(collection_name)

    for i in range(0, len(df), chunk_size):
        chunk_df = df.iloc[i:i + chunk_size].copy().reset_index(drop=True)
        print(f"\n📦 Processing chunk {i // chunk_size + 1}/{num_parts}: rows {i} to {i + chunk_size}")

        process_documents(
            df=chunk_df,
            DATA_PATH=DATA_PATH,
            bm25_texts_all=all_bm25_texts,
            product_ids_all=all_product_ids
        )

    # Only build BM25 once after all chunks
    print("\n📚 Creating final BM25 index...")
    if not bm25_index_exists(DATA_PATH):
        success = create_bm25_index(
            collection_name=collection_name,
            data_path=DATA_PATH,
            texts=all_bm25_texts,
            product_ids=all_product_ids
        )
        if not success:
            print("⚠️ Failed to create BM25 index. Continuing without it...")
    else:
        print("✅ BM25 index already exists. Skipping creation.")

In [26]:
#process_documents_in_chunks("products", df)
process_documents_in_chunks(df, DATA_PATH)


📦 Processing chunk 1/10: rows 0 to 5991
process_documents() has been triggered
Loaded 5991 products from chunk.
Creating structured text for embeddings...
Sample structured text: Title: Amazon Brand - Solimo Designer Lighting 3D Printed Hard Back Case Mobile Cover for Vivo V1 Max Amazon Brand - Solimo Designer Lighting 3D Printed Hard Back Case Mobile Cover for Vivo V1 Max Cat...
Generating embeddings...


Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Validating field lengths before insertion...
Truncating field value from 143 to 100 characters
Truncating field value from 109 to 100 characters
Truncating field value from 102 to 100 characters
Truncating field value from 4512 to 4000 characters
Inserting 5991 products into Milvus...
Data inserted successfully.
Creating index on embedding field...
Sample BM25 text: CELLULAR_PHONE_CASE CELLULAR_PHONE_CASE CELLULAR_PHONE_CASE  Amazon Brand - Solimo Designer Lighting 3D Printed Hard Back Case Mobile Cover for Vivo V1 Max Amazon Brand - Solimo Designer Lighting 3D P...

📦 Processing chunk 2/10: rows 5991 to 11982
process_documents() has been triggered
Loaded 5991 products from chunk.
Creating structured text for embeddings...
Sample structured text: Title: 365 by Whole Foods Market, Organic Tortilla Chips, Blue Corn, 12 Ounce 365 by Whole Foods Market, Organic Tortilla Chips, Blue Corn, 12 Ounce Category: GROCERY GROCERY GROCERY Features: Brought...
Generating embeddings...


Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Validating field lengths before insertion...
Truncating field value from 116 to 100 characters
Truncating field value from 109 to 100 characters
Truncating field value from 124 to 100 characters
Truncating field value from 125 to 100 characters
Truncating field value from 102 to 100 characters
Truncating field value from 103 to 100 characters
Truncating field value from 4562 to 4000 characters
Truncating field value from 4098 to 4000 characters
Truncating field value from 4505 to 4000 characters
Inserting 5991 products into Milvus...
Data inserted successfully.
Creating index on embedding field...
Sample BM25 text: GROCERY GROCERY GROCERY  365 by Whole Foods Market, Organic Tortilla Chips, Blue Corn, 12 Ounce 365 by Whole Foods Market, Organic Tortilla Chips, Blue Corn, 12 Ounce  365 by Whole Foods Market 365 by...

📦 Processing chunk 3/10: rows 11982 to 17973
process_documents() has been triggered
Loaded 5991 products from chunk.
Creating structured text for embeddings...
Sample struc

...

Validating field lengths before insertion...
Truncating field value from 208 to 200 characters
Truncating field value from 104 to 100 characters
Truncating field value from 125 to 100 characters
Truncating field value from 112 to 100 characters
Truncating field value from 169 to 100 characters
Truncating field value from 131 to 100 characters
Truncating field value from 128 to 100 characters
Inserting 5991 products into Milvus...
Data inserted successfully.
Creating index on embedding field...
Sample BM25 text: BOOT BOOT BOOT  Amazon Brand - find. Men's Classic Boots, Black (Black), 11 UK Amazon Brand - find. Men's Classic Boots, Black (Black), 11 UK  find. find.  nan nan...

📦 Processing chunk 9/10: rows 47928 to 53919
process_documents() has been triggered
Loaded 5991 products from chunk.
Creating structured text for embeddings...
Sample structured text: Title: Amazon Essentials Amelia Rain Boot, Black, UK 6 Amazon Essentials Amelia Rain Boot, Black, UK 6 Category: BOOT BOOT BOOT Fea

Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Validating field lengths before insertion...
Truncating field value from 127 to 100 characters
Truncating field value from 109 to 100 characters
Truncating field value from 162 to 100 characters
Truncating field value from 102 to 100 characters
Truncating field value from 128 to 100 characters
Truncating field value from 102 to 100 characters
Truncating field value from 116 to 100 characters
Truncating field value from 170 to 100 characters
Truncating field value from 4638 to 4000 characters
Truncating field value from 4776 to 4000 characters
Truncating field value from 4608 to 4000 characters
Inserting 5991 products into Milvus...
Data inserted successfully.
Creating index on embedding field...
Sample BM25 text: BOOT BOOT BOOT  Amazon Essentials Amelia Rain Boot, Black, UK 6 Amazon Essentials Amelia Rain Boot, Black, UK 6  Amazon Essentials Amazon Essentials  Black Black  Closed Toe Rubber shoe easy to slide ...

📦 Processing chunk 10/10: rows 53919 to 59910
process_documents() has be

Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Validating field lengths before insertion...
Truncating field value from 118 to 100 characters
Truncating field value from 112 to 100 characters
Truncating field value from 113 to 100 characters
Truncating field value from 112 to 100 characters
Truncating field value from 139 to 100 characters
Truncating field value from 102 to 100 characters
Truncating field value from 4600 to 4000 characters
Truncating field value from 4396 to 4000 characters
Inserting 5989 products into Milvus...
Data inserted successfully.
Creating index on embedding field...
Sample BM25 text: SHOES SHOES SHOES  Amazon Brand: Find mens slippers Amazon Brand: Find mens slippers  nan nan  nan nan...

📚 Creating final BM25 index...
NLTK resource 'punkt' is already available.
NLTK resource 'stopwords' is already available.
Starting BM25 tokenization...
Creating BM25 model with 59908 documents
BM25 model created successfully
BM25 index saved to /kaggle/working/bm25_index/productsFinal_bm25.pkl
